# Beginner 03: Secure Research Agent

**Level:** Beginner · **Duration:** 60–90 min · **Prerequisites:** Security Foundations (Beginner 01), Prompt Injection (Beginner 02)

## 1. Scenario and Objectives
This module builds directly upon Beginner 01 and Beginner 02. In this lesson, we build a credential-free enterprise research assistant. We will combine tool policy, provenance, and data sensitivity to demonstrate that **retrieved content = evidence**, not instructions or permission to act.

Our employee needs to ask the assistant internal questions (like retention policies). The corpus contains various document types (public, internal, confidential, poisoned). We will build a pipeline that safely retrieves, filters, generates, and validates answers.

In [ ]:
import sys, importlib
from pathlib import Path
for p in [Path("."), Path("curriculum/beginner/03-secure-research-agent")]:
    if (p / "03_secure_research_agent.py").exists():
        sys.path.insert(0, str(p.resolve()))
        break
lab = importlib.import_module("03_secure_research_agent")
print("Loaded module successfully.")

## 2. Threat Model
When building a research agent, several attacks are possible:
- **Prompt Injection**: A retrieved document contains instructions designed to hijack the model.
- **Data Exfiltration**: The model leaks confidential documents to unauthorized users.
- **Unauthorized Actions**: The model uses tools outside its allowed scope.
- **Citation Laundering**: The model fabricates citations to look authoritative while making unsupported claims.
- **Privilege Escalation**: An attacker attempts to forge their authorization context.

## 3. Metadata-Rich Corpus
Let's look at the documents available. Notice that they have explicit `Provenance` and `Sensitivity` metadata.

In [ ]:
for doc_id, doc in lab.DOCUMENT_STORE.items():
    print(f"{doc_id}: {doc.sensitivity.name} | {doc.provenance.name} | {doc.title}")

## 4. Why Heuristic Filtering is Insufficient
We have a detection heuristic to flag obvious injections. It helps observability, but attackers easily bypass these filters. Therefore, it **must never be the authorization boundary**.

In [ ]:
print("Obvious poison detected?", lab.detect_suspicious_content(lab.DOCUMENT_STORE["doc-poison-obvious"].text))
print("Bypass poison detected?", lab.detect_suspicious_content(lab.DOCUMENT_STORE["doc-poison-bypass"].text))

## 5. Least Privilege Capability Contract
The agent must operate under strict least privilege. The capability contract defines exactly what it is allowed to do.

In [ ]:
print("Allowed operations:", lab.ALLOWED_OPERATIONS)

## 6. Authoritative Access Context
A core enterprise principle is that the authorization context must come from an authoritative system (like an identity provider), not from the caller's request. `ResearchContext` is just a dataclass, not an authorization token. The real boundary is the `ResearchApplication` API, which only accepts a subject string and resolves the identity internally.

In [ ]:
app = lab.ResearchApplication()

# Let's see what happens if an unknown user tries to query the system:
resp = app.answer("unknown_eve", "What is the retention policy?")
audit = app.audit_sink.events[-1]
print(f"Unknown User State: {resp.terminal_state} ({audit.reason})")

# And let's see how Alice and Bob are resolved:
alice_ctx = lab.ResearchContextResolver.resolve("alice")
print(f"Alice authoritative access: {[s.name for s in alice_ctx.allowed_sensitivities]}")

## 7. Retrieval Authorization Check
Retrieval itself is an authorization boundary. We don't retrieve confidential data and hope the model hides it; we filter it *before* the model ever sees it. Let's see what happens when Alice queries for the Project Phoenix budget.

In [ ]:
auth_docs, blocked_docs = lab.RetrievalService.get_authorized_evidence("Project Phoenix budget", alice_ctx)
print(f"Authorized: {[d.document_id for d in auth_docs]}")
print(f"Blocked: {[d.document_id for d in blocked_docs]}")

## 8. Query Bounding
To prevent denial-of-service and context-window exhaustion, the system deterministically restricts the query length.

In [ ]:
resp = app.answer("alice", "a" * 600)
audit = app.audit_sink.events[-1]
print(f"State: {resp.terminal_state}, Reason: {audit.reason}")

## 9. Untrusted Model Output
Because prompt injection is possible, the model output (`ModelOutput`) is completely untrusted until validated by the deterministic application logic.

## 10. Deterministic Validators
The `PolicyEngine` enforces three deterministic checks:
1. **Capability**: Is the proposed action allowed?
2. **Citations**: If an answer is generated, does it cite at least one document? Do those citations exist and were they authorized for this user?
3. **Grounding**: Does the cited document actually support the generated claim? (Note: We use simulation fixtures here, but production requires entailment evaluation models).

## 11. Decoupling User Response from Internal Audit
We cleanly separate the safe data returned to the user (`ResearchResponse`) from the sensitive data logged for defenders (`AuditEvent`).

In [ ]:
resp = app.answer("alice", "ticket retention policy")
print("User Response:\n", resp, "\n")
audit = app.audit_sink.events[-1]
print("> Internal defender-only telemetry — not returned to the end user:")
print("Audit Event:\n", audit)

## 12. Poisoned Evidence Cannot Authorize Actions
If Alice's search retrieves the obvious poisoned document, the model might try to `reveal_secret`. But the Capability Contract blocks it.

In [ ]:
resp = app.answer("alice", "Read user profile")
audit = app.audit_sink.events[-1]
print(f"Terminal state: {resp.terminal_state} ({audit.reason})\nAnswer: {resp.answer}")

## 13. Poisoned Trusted Source
Even if a document's provenance is `INTERNAL`, it only serves as `INFORMATIONAL` evidence. It cannot grant operational authority. Here, a legacy document tries to authorize an email.

In [ ]:
resp = app.answer("alice", "Check legacy operations")
audit = app.audit_sink.events[-1]
print(f"Terminal state: {resp.terminal_state} ({audit.reason})\nAnswer: {resp.answer}")

## 14. Handling Insufficient Evidence
If no authorized evidence supports the query, we fail closed. The model is not allowed to hallucinate an answer without evidence.

In [ ]:
resp = app.answer("alice", "What color is the sky?")
audit = app.audit_sink.events[-1]
print(f"Terminal state: {resp.terminal_state} ({audit.reason})\nAnswer: {resp.answer}")

## 15. Secret Data Minimization & Audit Redaction
When Alice asks for the budget, the retrieval boundary blocks the confidential document. It never reaches the model, ensuring the secret is safe. 
Notice that the user-facing response does NOT reveal that a document was blocked, but the internal audit log tracks it! However, even the audit log redacts the actual secret value.

In [ ]:
resp = app.answer("alice", "What is the Project Phoenix budget?")
audit = app.audit_sink.events[-1]
print(f"User Answer: {resp.answer}")
print("> Internal defender-only telemetry — not returned to the end user:")
print(f"Audit Blocked Docs: {audit.blocked_document_ids}")
print(f"Secret leaked in audit? {'$4.2M' in str(audit.to_dict())}")


## 16. Validation: Citation Laundering
Citation laundering occurs when a model produces an incorrect/malicious claim, but cites a valid, trusted document to look authoritative.

In [ ]:
resp = app.answer("alice", "launder retention policy")
audit = app.audit_sink.events[-1]
print(f"Terminal state: {resp.terminal_state} ({audit.reason})\nAnswer: {resp.answer}")

## 17. Validation: Zero Citations
If the model answers but fails to cite evidence, we reject the answer deterministically.

In [ ]:
resp = app.answer("alice", "zero citation for retention policy")
audit = app.audit_sink.events[-1]
print(f"Terminal state: {resp.terminal_state} ({audit.reason})\nAnswer: {resp.answer}")

## 18. Adversarial Matrix
Run the complete demo to see how the system handles all edge cases safely.

In [ ]:
lab.run_demo()

## 19. Exercises
1. **New Sensitivity Tier:** Add a `RESTRICTED` sensitivity level and test access controls.
2. **New Prohibited Capability:** Add an `upload_file` tool and prove poisoned evidence cannot authorize it.
3. **Citation Validation:** Create an answer that cites a non-retrieved document. Ensure the system rejects it.
4. **Poisoned Trusted Source:** Prove that malicious text inside a trusted document cannot grant operational authority.
5. **Audit Redaction:** Ensure a synthetic confidential secret never appears in the audit output.

## 20. Production Upgrades
This simulation teaches boundaries. In production:
- The corpus is a Vector Database.
- Retrieval uses Semantic Search, and often enforces authorization inside the datastore.
- The Capability Contract is enforced by Policy-as-Code (e.g. OPA, Cedar).
- Grounding validation uses Entailment models (NLI) or specific claim-checking LLMs.